# Grok ResearchBot

**An AI-assisted research gap identification and technique proposal tool.**

This Google Colab notebook uses Retrieval-Augmented Generation (RAG) to:
1. Accept research-paper PDFs.
2. Extract and chunk their text.
3. Create embeddings and index the chunks in ChromaDB.
4. Retrieve literature passages relevant to a research question.
5. Ask a Groq-hosted language model to identify research gaps.
6. Propose one concrete technique grounded in the retrieved excerpts.

## Before you start

Each user must provide **their own Groq API key**. No API key is included in this notebook.

1. Create a key at [Groq Console – API Keys](https://console.groq.com/keys).
2. In Google Colab, open the **Secrets** panel (key icon in the left sidebar).
3. Add a secret named `GROQ_API_KEY`.
4. Paste your own key as its value and enable notebook access.
5. Run the cells from top to bottom.

This notebook is designed for **Google Colab**. The vector database is in-memory, so its contents are cleared when the runtime is reset.

## 1. Install dependencies

In [ ]:
!pip -q install -U pymupdf sentence-transformers chromadb groq

## 2. Import libraries and configure your personal Groq API key

In [ ]:
import os
import fitz  # PyMuPDF
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata, files

# Read the API key from this user's Google Colab Secrets.
# Never paste your private API key directly into a notebook cell.
api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY was not found. Open the Colab Secrets panel, "
        "add a secret named GROQ_API_KEY, paste your own Groq API key, "
        "and enable notebook access."
    )

groq_client = Groq(api_key=api_key)
print("Groq API client configured successfully.")

# Load the sentence-transformer embedding model.
# This model can run on CPU; a GPU is not required.
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

## 3. PDF text extraction and chunking

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract selectable text from a PDF using PyMuPDF."""
    text_parts = []

    with fitz.open(pdf_path) as doc:
        for page in doc:
            page_text = page.get_text("text")
            if page_text.strip():
                text_parts.append(page_text)

    return "\\n".join(text_parts)


def chunk_text(
    text: str,
    chunk_size: int = 500,
    overlap: int = 50
) -> list[str]:
    """Split text into word-based chunks with overlap."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size.")

    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk.strip():
            chunks.append(chunk)

    return chunks


print("PDF extraction and chunking functions are ready.")

## 4. Upload research papers and build the vector database

Upload one or more text-based PDF papers when prompted. Scanned PDFs that contain only images may require OCR; OCR is not included in this notebook.

In [ ]:
print("Upload one or more research-paper PDF files:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No PDF files were uploaded.")

# Create an in-memory ChromaDB collection for this Colab session.
chroma_client = chromadb.Client()

# Use a unique collection name in this runtime and remove an earlier
# collection with the same name if this cell is rerun.
collection_name = "research_papers"
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

all_chunks = []
chunk_ids = []
chunk_metadatas = []

for filename in uploaded.keys():
    if not filename.lower().endswith(".pdf"):
        print(f"Skipping non-PDF file: {filename}")
        continue

    print(f"Processing: {filename}")
    try:
        extracted_text = extract_text_from_pdf(filename)
    except Exception as exc:
        print(f"Could not read {filename}: {exc}")
        continue

    if not extracted_text.strip():
        print(
            f"No selectable text found in {filename}. "
            "It may be scanned or image-only; OCR is not included."
        )
        continue

    chunks = chunk_text(extracted_text)

    for chunk_index, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_ids.append(f"paper_{len(chunk_ids)}")
        chunk_metadatas.append({
            "source": filename,
            "chunk_index": chunk_index
        })

    print(f"  -> {len(chunks)} chunks")

if not all_chunks:
    raise ValueError(
        "No text chunks were created. Please upload readable, text-based PDF papers."
    )

print(f"Creating embeddings for {len(all_chunks)} chunks...")
embeddings = embed_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True
).tolist()

# Add in batches to avoid very large single requests.
batch_size = 100
for start in range(0, len(all_chunks), batch_size):
    end = start + batch_size
    collection.add(
        ids=chunk_ids[start:end],
        documents=all_chunks[start:end],
        embeddings=embeddings[start:end],
        metadatas=chunk_metadatas[start:end]
    )

print(f"Vector database built successfully: {collection.count()} chunks.")

## 5. Retrieve relevant literature excerpts

In [ ]:
def retrieve_context(query: str, top_k: int = 5) -> tuple[str, list[str]]:
    """Retrieve relevant text chunks and the names of their source PDFs."""
    if not query or not query.strip():
        raise ValueError("Please provide a non-empty research query.")

    if collection.count() == 0:
        raise ValueError("The vector database is empty. Upload and process PDFs first.")

    query_embedding = embed_model.encode([query]).tolist()
    number_to_retrieve = min(max(1, top_k), collection.count())

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=number_to_retrieve,
        include=["documents", "metadatas", "distances"]
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    context_parts = []
    for document, metadata in zip(documents, metadatas):
        source = metadata.get("source", "Unknown source")
        chunk_index = metadata.get("chunk_index", "Unknown")
        context_parts.append(
            f"Source: {source} | Chunk: {chunk_index}\\n{document}"
        )

    context = "\\n\\n---\\n\\n".join(context_parts)
    sources = sorted({
        metadata.get("source", "Unknown source")
        for metadata in metadatas
    })

    return context, sources


print("Literature retrieval function is ready.")

## 6. Research-gap identification and technique proposal

The prompts below preserve the notebook's original approach: identify gaps from retrieved excerpts, then propose one specific technique grounded in the provided literature.

In [ ]:
GAP_ANALYSIS_PROMPT = """You are a research analyst. Based on the following literature excerpts, identify explicit research gaps, limitations, or open problems.

Literature excerpts:
{context}

Please extract and summarize in the format:
1. **Gap Description**: Briefly describe the gap
2. **Source**: Which paper/excerpt it comes from
3. **Why It Is a Gap**: Briefly explain why this is a gap

If no explicit gaps are found in the excerpts, state "No explicit research gaps found in the provided excerpts."
Do not invent findings or claim that the excerpts establish something they do not say.
"""


TECHNIQUE_PROPOSAL_PROMPT = """You are a methodologist. You have identified the following research gap:

GAP: {gap_statement}

Here are the relevant literature excerpts that establish this gap and the current state of the art:
{context}

Your task is to propose ONE concrete algorithm or technique that could address this gap. Strictly follow these rules:

1. **Grounding**: The proposed technique must be an extension, combination, or modification of methods explicitly mentioned in the excerpts above. Do not invent entirely new fields.
2. **Specificity**: Do not say "use deep learning." Say "Use [specific architecture] with [specific modification] to handle [specific limitation]."
3. **Novelty check**: Before finalizing, verify whether this exact combination appears in the excerpts above. If it does, clearly state that the novelty may lie in applying it to this specific gap.
4. **Output format**:
   - **Proposed Technique Name**: [Short descriptive name]
   - **Core Idea**: [1-2 sentences]
   - **Implementation Sketch**: [3-4 bullet points]
   - **Why It Addresses the Gap**: [Directly map to the gap]

Important: You can only compare the proposal with the excerpts provided here. Do not claim that the technique is globally novel or that no prior work exists outside these excerpts.
"""


def call_llm(prompt: str, temperature: float = 0.3) -> str:
    """Send a prompt to the Groq-hosted model and return its response."""
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=2000,
    )

    content = response.choices[0].message.content
    return content.strip() if content else "The model returned an empty response."


print("Research prompts and Groq LLM function are ready.")

## 7. Run the complete research-assistance pipeline

Edit the research question below. The notebook retrieves relevant passages, identifies research gaps, and then proposes one technique. You can replace `gap_to_address` with a particular gap after reviewing the gap-analysis output.

In [ ]:
# Step 1: Enter a research question
research_query = "What are the limitations of current methods in this field?"

print("=" * 70)
print("STEP 1: RETRIEVING RELEVANT LITERATURE")
print("=" * 70)

context, sources = retrieve_context(research_query, top_k=5)

if not context.strip():
    raise ValueError(
        "No relevant excerpts were retrieved. Try a different research question."
    )

print(f"Retrieved excerpts from {len(sources)} source paper(s):")
for source in sources:
    print(f" - {source}")


# Step 2: Identify research gaps
print("\\n" + "=" * 70)
print("STEP 2: IDENTIFYING RESEARCH GAPS")
print("=" * 70)

gap_prompt = GAP_ANALYSIS_PROMPT.format(context=context)
gap_result = call_llm(gap_prompt)
print(gap_result)


# Step 3: Propose one concrete technique
# Review the gap-analysis result above. You may replace this value
# with one specific gap statement before running this cell again.
gap_to_address = gap_result

print("\\n" + "=" * 70)
print("STEP 3: PROPOSING A TECHNIQUE")
print("=" * 70)

proposal_prompt = TECHNIQUE_PROPOSAL_PROMPT.format(
    gap_statement=gap_to_address,
    context=context
)

proposal = call_llm(proposal_prompt, temperature=0.5)
print(proposal)

## Notes, limitations, and responsible use

- **API key:** Each user must supply their own `GROQ_API_KEY` through Colab Secrets. Never commit an API key to GitHub.
- **API usage:** Requests are made through the user's Groq account and are subject to Groq's current model availability, usage limits, and pricing.
- **Model configuration:** This notebook currently requests `openai/gpt-oss-120b`. If it is unavailable to your account, check Groq's current model documentation and update the `model` value in `call_llm()`.
- **Temporary storage:** ChromaDB is created in memory. Resetting or disconnecting the Colab runtime clears the indexed papers; upload and process them again in a new session.
- **PDF extraction:** The notebook extracts embedded/selectable PDF text. It does not perform OCR on scanned pages.
- **Research validity:** The identified gaps and proposed techniques are AI-generated suggestions. Review the original papers and conduct a broader literature search before making academic claims.
- **Novelty:** The notebook only compares the proposed technique with the retrieved excerpts. It cannot establish global research novelty.
- **Privacy:** Prompts sent to the language model are processed by Groq. Do not submit confidential or restricted documents unless you are authorized and have reviewed the applicable service terms.

## Troubleshooting

**`GROQ_API_KEY` not found:** Check that the Colab secret is named exactly `GROQ_API_KEY` and notebook access is enabled.

**Authentication or model error:** Verify your key and check which models are currently available to your Groq account.

**No text extracted from a PDF:** The file may be scanned/image-only. Use a text-based PDF or add an OCR workflow.

**Results disappear after reconnecting:** This is expected because the current ChromaDB collection is in-memory.

---

**Grok ResearchBot** — an experimental research-assistance notebook.